In [14]:
words=open("names.csv",encoding="utf-8").read().splitlines()

In [15]:
import re
import torch

clean_words = []

for w in words:

    w = re.sub(r"[^A-Za-z\s'& -]", "", w)
    w = " ".join(w.split())
    if w:
        clean_words.append(w)

words = clean_words
words[:8]

['lavera',
 'fairytabs',
 'arkive',
 'gaa',
 'cultiv cosmetique',
 'the green recipe',
 'mlanine',
 'base cosmetics']

In [16]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
print(itos)

{1: ' ', 2: '&', 3: "'", 4: '-', 5: 'a', 6: 'b', 7: 'c', 8: 'd', 9: 'e', 10: 'f', 11: 'g', 12: 'h', 13: 'i', 14: 'j', 15: 'k', 16: 'l', 17: 'm', 18: 'n', 19: 'o', 20: 'p', 21: 'q', 22: 'r', 23: 's', 24: 't', 25: 'u', 26: 'v', 27: 'w', 28: 'x', 29: 'y', 30: 'z', 0: '.'}


In [ ]:
#dataset creation

X,Y=[],[]
block_size=3 #context length i.e how many char to know to predict next one

for w in words[:1]:
    print(w)
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context),'--->',itos[ix])
        context=context[1:]+[ix]

X=torch.tensor(X)
Y=torch.tensor(Y)
        


lavera
... ---> l
..l ---> a
.la ---> v
lav ---> e
ave ---> r
ver ---> a
era ---> .


In [22]:
 #lookup table
C=torch.randn((31,2))
C[:5]

tensor([[-0.5667,  0.8839],
        [-0.9839,  0.0550],
        [ 0.5704,  1.5221],
        [-0.7433, -0.1897],
        [ 1.1275, -0.1012]])

In [25]:
emb=C[X]
emb[1]

tensor([[-0.5667,  0.8839],
        [-0.5667,  0.8839],
        [-1.5655, -0.9568]])

In [27]:
emb.size()

torch.Size([7, 3, 2])

In [ ]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1) #have to change is context len chnages therfore we use unbind

tensor([[-0.5667,  0.8839, -0.5667,  0.8839, -0.5667,  0.8839],
        [-0.5667,  0.8839, -0.5667,  0.8839, -1.5655, -0.9568],
        [-0.5667,  0.8839, -1.5655, -0.9568,  1.4559,  1.0993],
        [-1.5655, -0.9568,  1.4559,  1.0993,  0.9941,  0.7687],
        [ 1.4559,  1.0993,  0.9941,  0.7687,  0.9328, -0.9141],
        [ 0.9941,  0.7687,  0.9328, -0.9141,  0.2781,  1.4258],
        [ 0.9328, -0.9141,  0.2781,  1.4258,  1.4559,  1.0993]])

In [ ]:
torch.cat(torch.unbind(emb,1),1) # even though unbind creates a view but cat copies and allocate new memomry therefore we switch to view

tensor([[-0.5667,  0.8839, -0.5667,  0.8839, -0.5667,  0.8839],
        [-0.5667,  0.8839, -0.5667,  0.8839, -1.5655, -0.9568],
        [-0.5667,  0.8839, -1.5655, -0.9568,  1.4559,  1.0993],
        [-1.5655, -0.9568,  1.4559,  1.0993,  0.9941,  0.7687],
        [ 1.4559,  1.0993,  0.9941,  0.7687,  0.9328, -0.9141],
        [ 0.9941,  0.7687,  0.9328, -0.9141,  0.2781,  1.4258],
        [ 0.9328, -0.9141,  0.2781,  1.4258,  1.4559,  1.0993]])

In [32]:
emb.view(7,6)

tensor([[-0.5667,  0.8839, -0.5667,  0.8839, -0.5667,  0.8839],
        [-0.5667,  0.8839, -0.5667,  0.8839, -1.5655, -0.9568],
        [-0.5667,  0.8839, -1.5655, -0.9568,  1.4559,  1.0993],
        [-1.5655, -0.9568,  1.4559,  1.0993,  0.9941,  0.7687],
        [ 1.4559,  1.0993,  0.9941,  0.7687,  0.9328, -0.9141],
        [ 0.9941,  0.7687,  0.9328, -0.9141,  0.2781,  1.4258],
        [ 0.9328, -0.9141,  0.2781,  1.4258,  1.4559,  1.0993]])

In [40]:
W1=torch.randn((6,100))
b1=torch.randn(100)

In [ ]:

h=torch.tanh(emb.view(-1,6)@W1+b1) #check broadcasting rules
h

tensor([[-0.8872, -0.6870,  0.7657, -0.9551, -0.2946, -0.9977, -0.9011, -0.9979,
          0.8579, -0.9929, -0.6815, -0.9993, -0.9968,  0.9925,  0.8708, -0.9842,
          0.9957, -0.7234, -0.9986, -0.9967, -0.9951, -0.8576, -0.6204, -0.8923,
          0.9467, -0.2149,  0.9833, -0.9005,  0.8676,  0.8884,  0.2977, -0.9674,
         -0.5206,  0.6528, -0.2434, -0.9250,  0.9735, -0.8702, -0.1329, -0.5776,
         -0.9270,  0.3025,  0.9850, -0.8179,  0.9992,  0.6796,  0.9224,  0.0483,
          0.9288,  0.7356, -0.2835, -0.9858, -0.9612,  0.9831,  0.9913,  0.9416,
         -0.8115,  0.9161,  0.0260,  0.9999,  0.7721,  0.9909, -0.8648,  0.0423,
         -0.7235,  0.9943, -0.5274,  0.9971,  0.9978, -0.4987,  0.9999, -0.9807,
          0.7568, -0.8859, -0.9747,  0.7276, -0.0869, -0.8992,  0.9249,  0.7488,
          0.9902, -0.9850, -0.9124,  0.9821, -0.8414, -0.8458, -0.9874, -0.9589,
          0.9954,  0.0189,  0.4654,  0.9999, -0.2333, -0.8712, -0.9999, -0.7152,
         -0.5453,  0.7845, -

In [45]:
#last layer
W2=torch.randn((100,31))
b2=torch.randn(31)

In [48]:
logits=h@W2+b2
counts=logits.exp()
probs=counts/counts.sum(1,keepdims=True)

In [49]:
probs

tensor([[1.2400e-01, 7.3101e-16, 4.7999e-12, 3.2305e-10, 4.6159e-09, 2.5300e-02,
         6.5853e-14, 1.1170e-11, 9.2328e-06, 2.9034e-12, 3.8550e-09, 2.3133e-11,
         1.5458e-11, 5.0034e-13, 1.8942e-09, 4.0956e-16, 6.4644e-11, 8.5043e-01,
         5.9311e-10, 3.7030e-08, 6.1685e-09, 2.0219e-09, 3.2502e-05, 1.4240e-10,
         2.8610e-07, 7.5000e-09, 5.0318e-08, 1.1433e-04, 1.0996e-04, 1.7536e-11,
         4.6804e-10],
        [8.0479e-01, 3.6409e-07, 6.2367e-10, 1.3037e-09, 5.3514e-06, 3.6256e-05,
         1.1093e-06, 2.8570e-08, 4.3764e-05, 1.0970e-06, 1.0500e-12, 1.9201e-06,
         2.5049e-09, 2.1021e-13, 1.9197e-10, 2.7633e-09, 3.5489e-07, 1.1676e-06,
         3.9069e-07, 1.3221e-02, 1.0122e-14, 7.7419e-07, 3.3336e-08, 2.8731e-12,
         1.0353e-06, 1.8528e-14, 1.8317e-08, 2.5980e-07, 1.8189e-01, 4.7986e-14,
         2.4401e-09],
        [8.5558e-13, 1.7380e-16, 6.3849e-07, 1.3501e-09, 5.8588e-11, 1.0376e-03,
         2.5426e-13, 8.0689e-08, 1.9763e-11, 6.8414e-13, 2.1224e-

In [53]:
loss=-probs[torch.arange(7),Y].log().mean()
loss

tensor(14.9392)